# Demo: BERTweet Sentiment Inference (Seed 3 Comparison)
This notebook loads multiple sentiment checkpoints from Hugging Face, evaluates test accuracy, F1, recall, and precision on the Sentiment140 test split, and predicts sentiment for custom user text.

Models included in this demo:
- `BrianLeung/bertweet-sentiment140-full-finetune` (`seed3`)
- `BrianLeung/bertweet-sentiment140-lora` (`r32/seed3`, `r8/seed3`, `r4/seed3`)
- `BrianLeung/bertweet-sentiment140-prompt-tuning` (`seed3`)
- `BrianLeung/bertweet-sentiment140-classifier-only` (`seed3`)

Dataset:
- Load CSV from Hugging Face path: `BrianLeung/sentiment140-csv`
- Split into train, validate, test on `seed = 3`


In [ ]:
%pip uninstall -y torchao torchvision torchaudio
%pip install -U transformers datasets evaluate peft

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftConfig, PeftModel
from datasets import load_dataset
import evaluate
import torch
import re
import gc

# Set device for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model specifications for demo
MODEL_SPECS = [
    {
        "display_name": "full_finetune_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-full-finetune",
        "subfolder": "seed3",
        "model_type": "full",
    },
    {
        "display_name": "lora_r32_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r32/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "lora_r8_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r8/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "lora_r4_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-lora",
        "subfolder": "r4/seed3",
        "model_type": "peft",
    },
    {
        "display_name": "prompt_tuning_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-prompt-tuning",
        "subfolder": "seed3",
        "model_type": "peft",
    },
    {
        "display_name": "classifier_only_seed3",
        "repo": "BrianLeung/bertweet-sentiment140-classifier-only",
        "subfolder": "seed3",
        "model_type": "full",
    },
]

# Load base tokenizer for all models
def load_tokenizer_for_spec(spec):
    return AutoTokenizer.from_pretrained("vinai/bertweet-base")

# Load model and tokenizer for a given spec
def load_model_and_tokenizer(spec):
    tokenizer = load_tokenizer_for_spec(spec)

    if spec["model_type"] == "full":
        model = AutoModelForSequenceClassification.from_pretrained(
            spec["repo"],
            subfolder=spec["subfolder"],
            attn_implementation="eager",
        )
    elif spec["model_type"] == "peft":
        peft_config = PeftConfig.from_pretrained(
            spec["repo"],
            subfolder=spec["subfolder"],
        )
        base_model = AutoModelForSequenceClassification.from_pretrained(
            peft_config.base_model_name_or_path,
            num_labels=2,
            attn_implementation="eager",
        )
        model = PeftModel.from_pretrained(
            base_model,
            spec["repo"],
            subfolder=spec["subfolder"],
        )
    else:
        raise ValueError(f"Unknown model_type: {spec['model_type']}")

    # Ensure tokenizer and model vocab sizes match
    if hasattr(model, "get_input_embeddings"):
        embedding_layer = model.get_input_embeddings()
        if embedding_layer is not None:
            embedding_size = embedding_layer.num_embeddings
            tokenizer_size = len(tokenizer)
            if tokenizer_size > embedding_size:
                model.resize_token_embeddings(tokenizer_size)

    # Set pad token if missing
    if getattr(model.config, "pad_token_id", None) is None and getattr(tokenizer, "pad_token_id", None) is not None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Set attention implementation if available
    if hasattr(model.config, "_attn_implementation"):
        model.config._attn_implementation = "eager"

    model = model.to(device)
    model.eval()
    return model, tokenizer

In [ ]:
# Load and preprocess the Sentiment140 dataset (seed=3)
temp = load_dataset("BrianLeung/sentiment140-csv", encoding="ISO-8859-1")["train"]

temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])  # Keep only binary sentiment labels

# Relabel function: 0 -> 0 (negative), 4 -> 1 (positive)
def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

# Replace usernames and URLs with placeholders
def replace(text):
    text = re.sub(r"@\w+", "@USER", text)
    text = re.sub(r"http\S+|www\.\S+", "HTTPURL", text)
    return text

# Clean text in batch
def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]]}

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=3)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=3)

# Organize splits into a dictionary
dataset_dict = {
    "train": temp["train"].shuffle(seed=3),
    "test": test_valid["test"],
    "validation": test_valid["train"],
}

for split_name, split_data in dataset_dict.items():
    print(split_name, len(split_data))

In [ ]:
id2label = {0: "negative", 1: "positive"}

# Get number of virtual prompt tokens for a model (if any)
def get_prompt_virtual_tokens(model):
    peft_cfg = getattr(model, "peft_config", None)
    if not isinstance(peft_cfg, dict):
        return 0

    max_virtual_tokens = 0
    for cfg in peft_cfg.values():
        nvt = int(getattr(cfg, "num_virtual_tokens", 0) or 0)
        if nvt > max_virtual_tokens:
            max_virtual_tokens = nvt
    return max_virtual_tokens

# Get model input length info, accounting for prompt tokens
def get_model_length_info(model):
    max_positions = getattr(model.config, "max_position_embeddings", None)
    if max_positions is None:
        return 128, None, 0

    num_virtual_tokens = get_prompt_virtual_tokens(model)
    max_input_len = max_positions - 2 - num_virtual_tokens
    if max_input_len <= 0:
        raise ValueError(
            f"max_input_len={max_input_len} is invalid (max_positions={max_positions}, num_virtual_tokens={num_virtual_tokens})."
        )
    return max_input_len, max_positions, num_virtual_tokens

def get_max_input_len(model):
    max_input_len, _, _ = get_model_length_info(model)
    return max_input_len

# Predict sentiment for a batch of texts
def predict_batch(model, tokenizer, texts, max_length):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    # Check token and sequence length validity
    if "input_ids" in encoded:
        max_token_id = int(encoded["input_ids"].max().item())
        min_token_id = int(encoded["input_ids"].min().item())
        vocab_size = model.get_input_embeddings().num_embeddings
        if min_token_id < 0 or max_token_id >= vocab_size:
            raise ValueError(
                f"Token id range [{min_token_id}, {max_token_id}] is invalid for vocab size {vocab_size}."
            )

        seq_len = int(encoded["input_ids"].shape[1])
        max_positions = getattr(model.config, "max_position_embeddings", None)
        num_virtual_tokens = get_prompt_virtual_tokens(model)
        if max_positions is not None and (seq_len + 2 + num_virtual_tokens) > max_positions:
            raise ValueError(
                f"Sequence too long for model: seq_len={seq_len}, num_virtual_tokens={num_virtual_tokens}, "
                f"max_positions={max_positions}."
            )

    encoded = encoded.to(device)

    with torch.no_grad():
        logits = model(**encoded).logits
        preds = torch.argmax(logits, dim=-1)
    return preds.cpu().tolist()

# Compute binary metric (accuracy, f1, recall, precision)
def compute_binary_metric(metric, metric_name, predictions, references):
    common_kwargs = {
        "predictions": predictions,
        "references": references,
        "average": "binary",
        "pos_label": 1,
    }
    try:
        return metric.compute(**common_kwargs, zero_division=0)[metric_name]
    except TypeError:
        return metric.compute(**common_kwargs)[metric_name]

# Evaluate all models on the test set
test_ds = dataset_dict["test"]
batch_size = 64
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
recall_metric = evaluate.load("recall")
precision_metric = evaluate.load("precision")
model_results = []

for spec in MODEL_SPECS:
    print(
        f"Evaluating {spec['display_name']} from {spec['repo']} ({spec['subfolder']})"
    )
    model, tokenizer = load_model_and_tokenizer(spec)
    max_input_len, max_positions, num_virtual_tokens = get_model_length_info(model)
    if max_positions is not None:
        print(
            f"  max_input_len={max_input_len} (max_positions={max_positions}, virtual_tokens={num_virtual_tokens})"
        )

    all_preds = []
    for i in range(0, len(test_ds), batch_size):
        batch_texts = test_ds[i:i + batch_size]["Text"]
        batch_preds = predict_batch(
            model,
            tokenizer,
            batch_texts,
            max_length=max_input_len,
        )
        all_preds.extend(batch_preds)

    references = test_ds["Label"]
    metric_values = {
        "accuracy": accuracy_metric.compute(
            predictions=all_preds,
            references=references,
        )["accuracy"],
        "f1": compute_binary_metric(
            f1_metric,
            "f1",
            all_preds,
            references,
        ),
        "recall": compute_binary_metric(
            recall_metric,
            "recall",
            all_preds,
            references,
        ),
        "precision": compute_binary_metric(
            precision_metric,
            "precision",
            all_preds,
            references,
        ),
    }

    model_results.append(
        {
            "display_name": spec["display_name"],
            "repo": spec["repo"],
            "subfolder": spec["subfolder"],
            **metric_values,
        }
    )
    print(
        f"  accuracy={metric_values['accuracy']:.4f} | "
        f"f1={metric_values['f1']:.4f} | "
        f"recall={metric_values['recall']:.4f} | "
        f"precision={metric_values['precision']:.4f}"
    )

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nSummary (sorted by accuracy):")
for row in sorted(model_results, key=lambda x: x["accuracy"], reverse=True):
    print(
        f"- {row['display_name']} ({row['subfolder']}): "
        f"accuracy={row['accuracy']:.4f}, "
        f"f1={row['f1']:.4f}, "
        f"recall={row['recall']:.4f}, "
        f"precision={row['precision']:.4f}"
    )

# Sentiment prediction on custom text
Input custom data at specified location to see model predictions.

In [ ]:
# Predict sentiment for a custom text using all models
def classify_text_all_models(text):
    cleaned = replace(text)
    predictions = []

    for spec in MODEL_SPECS:
        model, tokenizer = load_model_and_tokenizer(spec)
        max_input_len = get_max_input_len(model)
        pred_id = predict_batch(
            model,
            tokenizer,
            [cleaned],
            max_length=max_input_len,
        )[0]
        label = id2label[pred_id]

        predictions.append(
            {
                "model": spec["display_name"],
                "repo": spec["repo"],
                "subfolder": spec["subfolder"],
                "predicted_label": label,
                "predicted_id": pred_id,
            }
        )

        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "text": text,
        "cleaned_text": cleaned,
        "predictions": predictions,
    }

#------------------ User input --------------------------
user_text = "I love how fast and reliable this model is!"
#--------------------------------------------------------
result = classify_text_all_models(user_text)

print(f"Text: {result['text']}")
print(f"Cleaned text: {result['cleaned_text']}")
print("\nPredictions:")
for item in result["predictions"]:
    print(
        f"- {item['model']} ({item['subfolder']}): {item['predicted_label']} ({item['predicted_id']})"
    )